In [ ]:
# Cell 1: imports and project-root setup
import sys
from pathlib import Path

def find_project_root(start: Path) -> Path:
    start = start.resolve()
    candidates = [start] + list(start.parents)
    for candidate in candidates:
        if (candidate / "src").exists():
            return candidate
    return start

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
import gym
import d4rl  # noqa: F401

from src.experiment_config import *
from src.config import CHECKPOINTS_DIR, RAW_METRICS_DIR, OBS_STATS_DIR
from src.dataset import NoisyOfflineRLDataset
from src.linear_encoder import LinearEncoder
from src.iql import IQLAgent
from src.train_eval import (
    eval_policy_on_env,
    load_and_freeze_encoder,
    save_metrics_json,
    train_iql_from_loader,
)

In [ ]:
# Cell 2: experiment configuration
# PPF-framework variant of PCA-IQL: replaces unsupervised PCA with a
# supervised linear layer trained against privileged clean states.
METHOD = "linear_iql"

def noise_tag(noise_dim, noise_scale, noise_type):
    ns = str(noise_scale).replace(".", "p")
    return f"nd{noise_dim}_ns{ns}_{noise_type}"

NOISE_TAG = noise_tag(NOISE_DIM, NOISE_SCALE, NOISE_TYPE)
SEED_TAG  = f"seed_{SEED}"

CKPT_DIR    = CHECKPOINTS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
METRICS_DIR = RAW_METRICS_DIR / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG
OBS_DIR     = OBS_STATS_DIR   / METHOD / ENV_NAME / NOISE_TAG / SEED_TAG

for d in [CKPT_DIR, METRICS_DIR, OBS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DEVICE:", DEVICE)
print("METHOD:", METHOD)

In [ ]:
# Cell 3: dataset and dataloader
dataset = NoisyOfflineRLDataset(
    env_name=ENV_NAME,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    seed=SEED,
    use_timeouts=True,
    noise_type=NOISE_TYPE,
)

train_loader = DataLoader(
    dataset, batch_size=BATCH_SIZE, shuffle=True, drop_last=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

state_dim      = dataset.noisy_obs.shape[1]
action_dim     = dataset.actions.shape[1]
true_state_dim = dataset.obs_dim

# Linear encoder projects into clean-state space: latent_dim = true_state_dim.
LATENT_DIM = true_state_dim

np.savez(
    OBS_DIR / "obs_stats.npz",
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
)
print("state_dim (noisy):", state_dim)
print("true_state_dim:", true_state_dim)
print("action_dim:", action_dim)
print("latent_dim:", LATENT_DIM)

In [ ]:
# Cell 4: pretrain the supervised linear encoder
# Objective: minimize MSE(W * noisy_obs, pure_obs)
# Uses the same privileged clean-state signal as PlainEncoder /
# DisentangledEncoder, but with a strictly linear model — isolating
# the contribution of encoder nonlinearity within the PPF framework.

torch.manual_seed(SEED)
np.random.seed(SEED)

encoder   = LinearEncoder(state_dim=state_dim, latent_dim=LATENT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

pretrain_loader = DataLoader(
    dataset, batch_size=PRETRAIN_BS, shuffle=True, drop_last=True,
    num_workers=4, pin_memory=True, persistent_workers=True,
)

for epoch in range(1, PRETRAIN_EPOCHS + 1):
    encoder.train()
    losses = []

    for obs, act, next_obs, rew, done, pure_obs, pure_next_obs in pretrain_loader:
        obs      = obs.to(DEVICE)
        pure_obs = pure_obs.to(DEVICE)

        z, _ = encoder(obs)
        loss  = F.mse_loss(z, pure_obs)

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        losses.append(float(loss.item()))

    print(f"[pretrain] epoch {epoch}, loss={np.mean(losses):.4f}")

CKPT_ENCODER = CKPT_DIR / f"encoder_epoch_{PRETRAIN_EPOCHS}.pth"
torch.save(encoder.state_dict(), CKPT_ENCODER)
print("Saved encoder:", CKPT_ENCODER)

In [ ]:
# Cell 5: load and freeze encoder, then train IQL
encoder = load_and_freeze_encoder(
    encoder=encoder,
    ckpt_path=CKPT_ENCODER,
    device=DEVICE,
)

iql = IQLAgent(
    latent_dim=LATENT_DIM,
    action_dim=action_dim,
    device=DEVICE,
    expectile=0.7,
    temperature=3.0,
    discount=0.99,
)

iql_history = train_iql_from_loader(
    iql=iql,
    train_loader=train_loader,
    device=DEVICE,
    epochs=EPOCHS,
    ckpt_dir=CKPT_DIR,
    method=METHOD,
    save_every=10,
    encoder=encoder,
    repr_mode="linear",
    use_tqdm=False,
)

In [ ]:
# Cell 6: evaluate and save metrics
print("Start evaluating ...")
metrics = eval_policy_on_env(
    iql=iql,
    env_name=ENV_NAME,
    encoder=encoder,
    method=METHOD,
    obs_mean=dataset.obs_mean,
    obs_std=dataset.obs_std,
    true_state_dim=true_state_dim,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    episodes=20,
    max_steps=1000,
    seed=SEED,
    device=DEVICE,
    use_fixed_noise=True,
)

metrics_path = save_metrics_json(
    metrics_dir=METRICS_DIR,
    metrics=metrics,
    env_name=ENV_NAME,
    method=METHOD,
    seed=SEED,
    noise_dim=NOISE_DIM,
    noise_scale=NOISE_SCALE,
    noise_type=NOISE_TYPE,
    extra={
        "latent_dim": LATENT_DIM,
        "pretrain_epochs": PRETRAIN_EPOCHS,
        "iql_epochs": EPOCHS,
    },
)
print(metrics)
print("Saved metrics:", metrics_path)